# Whitelist Customer Visualization

This notebook follows the workflow below:

1. Load `data_df` and `pred_df`.
2. Remove customers in tiers `A`, `B`, and `C`, and keep the same customer universe in `pred_df`.
3. Split customers in `pred_df` into `id1` (top 10%) and `id2` (remaining 90%).
4. Compare `id1` and `id2` with tier-level boxplots.
5. Compare tier, gender, and age using ID1 selection rates that control for unequal group sizes.
6. Compare the remaining demographic and continuous-variable distributions of `id1` and `id2`.
7. Export descriptive statistics.

If `pred_probability` exists in `pred_df`, the data are sorted from highest to lowest prediction probability before the top-10% split.


## 1. Imports and plotting settings

### Compatibility note

This notebook avoids `sns.histplot()` and uses `matplotlib.pyplot.hist()` for histogram-based visualizations, so it can run with older seaborn versions.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["font.sans-serif"] = ['SimHei','DejaVu Sans']
plt.rcParams["axes.unicode_minus"] = False

#sns.set_theme(style="whitegrid")


## 2. File paths and output directory

In [ ]:
# File paths
DATA_PATH = "whitelist0810.csv"
PRED_PATH = "whitelist0810_y_freq_probability.csv"

# Output directory
OUTPUT_DIR = "visualization_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ID_COL = "cst_id"
SCORE_COL = "pred_probability"


## 3. Load data and remove tiers A, B, and C

In [ ]:
data_df = pd.read_csv(DATA_PATH)
pred_df = pd.read_csv(PRED_PATH)

# Check the customer ID column
if ID_COL not in data_df.columns:
    raise KeyError(f"{ID_COL} is not found in data_df.")

if ID_COL not in pred_df.columns:
    raise KeyError(f"{ID_COL} is not found in pred_df.")

if "tier" not in data_df.columns:
    raise KeyError("tier is not found in data_df.")

# Use the same ID type in both datasets
data_df[ID_COL] = data_df[ID_COL].astype(str)
pred_df[ID_COL] = pred_df[ID_COL].astype(str)

# Remove tiers A, B, and C from data_df
excluded_tiers = ["A", "B", "C"]
data_df = data_df[
    ~data_df["tier"].astype(str).str.strip().str.upper().isin(excluded_tiers)
].copy()

# Keep only customers that remain in data_df
cst_id_list = data_df[ID_COL].drop_duplicates().tolist()
pred_df = pred_df[pred_df[ID_COL].isin(cst_id_list)].copy()

# If pred_df itself also contains tier, remove A/B/C there as an additional safeguard
if "tier" in pred_df.columns:
    pred_df = pred_df[
        ~pred_df["tier"].astype(str).str.strip().str.upper().isin(excluded_tiers)
    ].copy()

print("data_df shape after tier filtering:", data_df.shape)
print("pred_df shape after tier filtering:", pred_df.shape)

display(data_df.head())
display(pred_df.head())


## 4. Split customers into ID1 (top 10%) and ID2 (remaining 90%)

In [ ]:
# Remove duplicate customer IDs while preserving one row per customer
pred_unique = pred_df.drop_duplicates(subset=[ID_COL]).copy()

# If prediction probability exists, rank customers from highest to lowest probability
if SCORE_COL in pred_unique.columns:
    pred_unique[SCORE_COL] = pd.to_numeric(pred_unique[SCORE_COL], errors="coerce")
    pred_unique = pred_unique.sort_values(
        SCORE_COL,
        ascending=False,
        na_position="last"
    ).reset_index(drop=True)
else:
    print(
        f"Warning: {SCORE_COL} is not found in pred_df. "
        "The current row order will be used for the 10% split."
    )

pred_ids = pred_unique[ID_COL].tolist()

cutoff = int(np.ceil(len(pred_ids) * 0.10))

id1 = pred_ids[:cutoff]
id2 = pred_ids[cutoff:]

print("Total customers in pred_df:", len(pred_ids))
print("ID1 customers (top 10%):", len(id1))
print("ID2 customers (remaining 90%):", len(id2))
print("\nFirst 10 customer IDs in ID1:")
print(id1[:10])


## 5. Add the ID1 / ID2 group label to `data_df`

In [ ]:
plot_df = data_df.copy()

plot_df["group"] = np.select(
    [
        plot_df[ID_COL].isin(id1),
        plot_df[ID_COL].isin(id2),
    ],
    [
        "id1",
        "id2",
    ],
    default="other",
)

# Keep only customers appearing in pred_df
plot_df = plot_df[
    plot_df["group"].isin(["id1", "id2"])
].copy()

print("Matched customers in data_df:")
display(plot_df["group"].value_counts().to_frame("count"))


## 6. Standardize indicator column names

In [ ]:
rename_dict = {
    "YR_ACM_MPB_FNCLTX_AMT": "yr_acm_mpb_fncltx_amt",
    "YR_ACM_MPB_LAND_MONUM": "yr_acm_mpb_land_monum",
    "ACGMOCLRR6MAMPBFTXAMT": "acgmoclrr6mampbftxamt",
}

plot_df = plot_df.rename(columns=rename_dict)

print("Available columns:")
print(plot_df.columns.tolist())


In [ ]:
# ============================================================
# 由成为我行客户时间生成“成为我行客户天数”
# 原始日期格式示例：25JUL2016
# ============================================================

BECOME_DATE_COL = "bank_cust_become_date"
BECOME_DAYS_COL = "days_since_become_cust"

# 与模型预处理代码保持相同口径
SNAPSHOT_DATE = pd.Timestamp("2026-06-24")

if BECOME_DATE_COL not in plot_df.columns:
    print(
        f"无法生成 {BECOME_DAYS_COL}："
        f"数据中不存在 {BECOME_DATE_COL}"
    )
else:
    # 清理日期字符串：
    # 例如把“25jul2016”“ 25JUL2016 ”统一为“25JUL2016”
    become_date_raw = (
        plot_df[BECOME_DATE_COL]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    # 将空字符串及常见缺失字符统一设为缺失
    become_date_raw = become_date_raw.mask(
        become_date_raw.isna()
        | become_date_raw.isin(["", "NAN", "NONE", "NAT", "<NA>"])
    )

    # 按25JUL2016格式解析
    plot_df[BECOME_DATE_COL] = pd.to_datetime(
        become_date_raw,
        format="%d%b%Y",
        errors="coerce"
    )

    # 计算截至快照日期的客户关系持续天数
    plot_df[BECOME_DAYS_COL] = (
        SNAPSHOT_DATE
        - plot_df[BECOME_DATE_COL]
    ).dt.days

    # 如果成为客户时间晚于快照日期，将客户时长设为缺失
    future_mask = (
        plot_df[BECOME_DAYS_COL] < 0
    )
    future_count = int(future_mask.sum())

    if future_count > 0:
        print(
            f"发现 {future_count} 名客户的成为客户时间"
            "晚于快照日期，已将客户时长设为缺失。"
        )
        plot_df.loc[future_mask, BECOME_DAYS_COL] = np.nan

    # 输出日期解析情况
    original_non_missing = int(become_date_raw.notna().sum())
    parsed_count = int(plot_df[BECOME_DATE_COL].notna().sum())
    parse_failed_count = original_non_missing - parsed_count

    print("成为我行客户时间解析完成")
    print(f"原始非空日期数：{original_non_missing:,}")
    print(f"成功解析日期数：{parsed_count:,}")
    print(f"解析失败日期数：{parse_failed_count:,}")
    print(f"快照日期：{SNAPSHOT_DATE.date()}")
    print(
        f"{BECOME_DAYS_COL}有效值数："
        f"{plot_df[BECOME_DAYS_COL].notna().sum():,}"
    )

    # 查看解析结果
    display(
        pd.DataFrame({
            "原始成为客户时间": become_date_raw,
            "解析后成为客户时间": plot_df[BECOME_DATE_COL],
            "成为我行客户时长（天）": plot_df[BECOME_DAYS_COL],
        }).head(20)
    )

    # 查看客户时长描述统计
    display(
        plot_df[BECOME_DAYS_COL]
        .describe()
        .rename("成为我行客户时长（天）")
        .to_frame()
    )


In [ ]:
# ============================================================
# 客户星级分析：
# 1. Top 10%潜在营销客户 vs 其余90%客户
# 2. 分人才等级展示每个客户星级的人数占比
# ============================================================
STAR_COL = "cst_star_cd"
TIER_COL = "tier"

GROUP_LABEL_MAP_CN = {
    "id1": "Top 10%潜在营销客户",
    "id2": "其余90%非潜在营销客户",
}
required_cols = [ID_COL, "group", STAR_COL, TIER_COL]
missing_cols = [col for col in required_cols if col not in plot_df.columns]

if missing_cols:
    print("无法生成客户星级图，缺少字段：", missing_cols)
else:
    # 每个客户只保留一行，防止一个客户多行导致重复统计
    star_df = plot_df[required_cols].copy()
    star_df = star_df.drop_duplicates(subset=[ID_COL], keep="first")

    # 清洗客户星级
    star_df[STAR_COL] = (
        star_df[STAR_COL].fillna("缺失").astype(str).str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    # 清洗人才等级
    star_df[TIER_COL] = (
        star_df[TIER_COL].fillna("缺失").astype(str).str.strip().str.upper()
    )
    star_df["客户分组"] = star_df["group"].map(GROUP_LABEL_MAP_CN)

    print("用于客户星级分析的客户数：", len(star_df))
    print("\n分组客户数：")
    display(star_df["客户分组"].value_counts().to_frame("客户数"))

    # ============================================================
    # 图1：Top 10%与其余90%的客户星级分布对比
    # 口径：每组内部各星级客户占比
    # ============================================================
    star_group_count = (
        star_df.groupby(["客户分组", STAR_COL])[ID_COL]
        .nunique().rename("客户数").reset_index()
    )
    star_group_count["人数占比"] = (
        star_group_count["客户数"]
        / star_group_count.groupby("客户分组")["客户数"].transform("sum")
        * 100
    )
    print("Top 10%潜在客户与其余90%客户的星级分布：")
    display(star_group_count)

    plt.figure(figsize=(11, 6))
    ax = sns.barplot(
        data=star_group_count, x=STAR_COL, y="人数占比", hue="客户分组",
        hue_order=["Top 10%潜在营销客户", "其余90%非潜在营销客户"],
    )
    ax.set_xlabel("客户星级代码")
    ax.set_ylabel("组内客户人数占比（%）")
    ax.set_title("Top 10%潜在营销客户与其余90%客户的星级分布")
    ax.legend(title="客户分组")

    # 兼容旧版Matplotlib，不能使用ax.bar_label()
    for patch in ax.patches:
        height = patch.get_height()
        if pd.notna(height) and height > 0:
            ax.text(
                patch.get_x() + patch.get_width() / 2, height, f"{height:.1f}%",
                ha="center", va="bottom", fontsize=8,
            )

    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "客户星级分布_潜在与非潜在客户对比.png"),
        dpi=300, bbox_inches="tight",
    )
    plt.show()
    plt.close()


## Customer counts by tier and group

The table below reports the number of unique customers in ID1 and ID2 within each tier, together with the total customer count.


In [ ]:
# ============================================================
# Customer count by tier and group
# ============================================================

tier_group_count = (
    plot_df
    .groupby(["tier", "group"])[ID_COL]
    .nunique()
    .unstack(fill_value=0)
)

# Ensure ID1 and ID2 columns are present and ordered
for col in ["id1", "id2"]:
    if col not in tier_group_count.columns:
        tier_group_count[col] = 0

tier_group_count = tier_group_count[["id1", "id2"]]

# Add total number of unique customers within each tier
tier_group_count["Total"] = tier_group_count.sum(axis=1)

# Rename columns for display
tier_group_count = tier_group_count.rename(
    columns={
        "id1": "ID1: Top 10%",
        "id2": "ID2: Remaining 90%",
    }
)

display(tier_group_count)


## ID1 selection rate by tier

This visualization controls for differences in the original size of each tier. For each tier, it reports the share of customers classified into ID1 (Top 10%). The dashed horizontal line shows the overall ID1 share in the full sample.


In [ ]:
# ============================================================
# ID1 selection rate by tier
# ============================================================

# Count unique customers by tier and group
tier_selection = (
    plot_df
    .groupby(["tier", "group"])[ID_COL]
    .nunique()
    .unstack(fill_value=0)
)

# Ensure both columns exist
for col in ["id1", "id2"]:
    if col not in tier_selection.columns:
        tier_selection[col] = 0

tier_selection = tier_selection[["id1", "id2"]]

# Total number of customers within each tier
tier_selection["Total"] = (
    tier_selection["id1"]
    + tier_selection["id2"]
)

# Share of each tier entering ID1
tier_selection["ID1_rate"] = (
    tier_selection["id1"]
    / tier_selection["Total"]
)

# Actual overall ID1 rate in the analysis sample
overall_id1_rate = (
    plot_df.loc[
        plot_df["group"] == "id1",
        ID_COL
    ].nunique()
    / plot_df[ID_COL].nunique()
)

# Arrange common tier labels in a natural order
preferred_order = ["D", "E", "F1", "F2", "F3"]

available_tiers = [
    tier
    for tier in preferred_order
    if tier in tier_selection.index
]

remaining_tiers = [
    tier
    for tier in tier_selection.index
    if tier not in available_tiers
]

tier_order = available_tiers + sorted(remaining_tiers)

tier_plot = (
    tier_selection
    .loc[tier_order]
    .reset_index()
)

# Display the underlying statistics
tier_rate_table = tier_plot[
    [
        "tier",
        "id1",
        "id2",
        "Total",
        "ID1_rate"
    ]
].copy()

tier_rate_table["ID1_rate_percent"] = (
    tier_rate_table["ID1_rate"] * 100
)

display(tier_rate_table)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    tier_plot["tier"],
    tier_plot["ID1_rate"] * 100
)

# Overall benchmark
ax.axhline(
    y=overall_id1_rate * 100,
    linestyle="--",
    linewidth=2,
    label=f"Overall ID1 rate ({overall_id1_rate * 100:.1f}%)"
)

ax.set_xlabel("Tier")
ax.set_ylabel("Customers in ID1 (%)")
ax.set_title("ID1 Selection Rate by Tier")

# Add percentage labels above bars
for bar, rate in zip(
    bars,
    tier_plot["ID1_rate"] * 100
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{rate:.1f}%",
        ha="center",
        va="bottom"
    )

ax.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "id1_selection_rate_by_tier.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()


## ID1 selection rate by gender and age group

These figures use the same logic as the tier selection-rate figure. For each gender or age group, the plotted value is the share of customers who enter ID1. This controls for differences in the original size of each group.


In [ ]:
# ============================================================
# ID1 selection rate by Gender and Age Group
# ============================================================

def plot_id1_selection_rate(
    df,
    category_col,
    xlabel,
    title,
    output_name,
    category_order=None,
):
    """
    Plot P(ID1 | category) using unique customers.

    This controls for unequal original category sizes because the denominator
    is the total number of customers within each category.
    """

    required_cols = [ID_COL, category_col, "group"]
    missing = [col for col in required_cols if col not in df.columns]

    if missing:
        print(f"Skip {category_col}; missing columns: {missing}")
        return None

    temp = df[required_cols].copy()

    # Count each customer once
    temp = temp.drop_duplicates(subset=[ID_COL], keep="first")

    temp[category_col] = (
        temp[category_col]
        .fillna("Missing")
        .astype(str)
    )

    selection = (
        temp
        .groupby([category_col, "group"])[ID_COL]
        .nunique()
        .unstack(fill_value=0)
    )

    for col in ["id1", "id2"]:
        if col not in selection.columns:
            selection[col] = 0

    selection = selection[["id1", "id2"]]

    selection["Total"] = (
        selection["id1"]
        + selection["id2"]
    )

    selection["ID1_rate"] = np.where(
        selection["Total"] > 0,
        selection["id1"] / selection["Total"],
        np.nan
    )

    # Use the actual ID1 share among observations included in this analysis
    overall_rate = (
        temp.loc[temp["group"] == "id1", ID_COL].nunique()
        / temp[ID_COL].nunique()
    )

    if category_order is not None:
        available = [
            category
            for category in category_order
            if category in selection.index
        ]

        remaining = [
            category
            for category in selection.index
            if category not in available
        ]

        selection = selection.loc[
            available + sorted(remaining)
        ]

    selection_table = selection.reset_index().copy()
    selection_table["ID1_rate_percent"] = (
        selection_table["ID1_rate"] * 100
    )

    display(selection_table)

    fig, ax = plt.subplots(figsize=(10, 6))

    bars = ax.bar(
        selection_table[category_col].astype(str),
        selection_table["ID1_rate"] * 100
    )

    ax.axhline(
        y=overall_rate * 100,
        linestyle="--",
        linewidth=2,
        label=f"Overall ID1 rate ({overall_rate * 100:.1f}%)"
    )

    for bar, rate in zip(
        bars,
        selection_table["ID1_rate"] * 100
    ):
        if pd.notna(rate):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{rate:.1f}%",
                ha="center",
                va="bottom"
            )

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Customers in ID1 (%)")
    ax.set_title(title)
    ax.legend()

    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            output_name
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    return selection_table


# ------------------------------------------------------------
# Gender
# ------------------------------------------------------------

gender_selection_table = plot_id1_selection_rate(
    df=plot_df,
    category_col="gnd_cd",
    xlabel="Gender",
    title="ID1 Selection Rate by Gender",
    output_name="id1_selection_rate_by_gender.png",
)


# ------------------------------------------------------------
# Age groups
# ------------------------------------------------------------

if "age" not in plot_df.columns:
    print("Skip age; column not found.")

else:
    age_selection_df = plot_df[
        [ID_COL, "age", "group"]
    ].copy()

    age_selection_df["age"] = pd.to_numeric(
        age_selection_df["age"],
        errors="coerce"
    )

    age_selection_df = age_selection_df.dropna(
        subset=["age"]
    )

    age_selection_df["age_group"] = pd.cut(
        age_selection_df["age"],
        bins=[0, 25, 35, 45, 55, 65, np.inf],
        labels=[
            "25 or below",
            "26-35",
            "36-45",
            "46-55",
            "56-65",
            "Above 65",
        ],
        include_lowest=True
    )

    age_selection_df["age_group"] = (
        age_selection_df["age_group"]
        .astype(str)
    )

    age_group_order = [
        "25 or below",
        "26-35",
        "36-45",
        "46-55",
        "56-65",
        "Above 65",
    ]

    age_selection_table = plot_id1_selection_rate(
        df=age_selection_df,
        category_col="age_group",
        xlabel="Age Group",
        title="ID1 Selection Rate by Age Group",
        output_name="id1_selection_rate_by_age_group.png",
        category_order=age_group_order,
    )


## 7. Function: tier-level boxplots for ID1 vs ID2

In [ ]:
GROUP_LABEL_MAP = {
    "id1": "ID1: Top 10%",
    "id2": "ID2: Remaining 90%",
}

def plot_boxplot_by_tier(
    df,
    value_col,
    ylabel,
    title=None,
    output_name=None,
):
    required_cols = ["tier", "group", value_col]
    missing = [col for col in required_cols if col not in df.columns]

    if missing:
        print(f"Skip {value_col}; missing columns: {missing}")
        return

    temp = df[["tier", "group", value_col]].copy()

    temp[value_col] = pd.to_numeric(
        temp[value_col],
        errors="coerce",
    )

    temp = temp.dropna(
        subset=["tier", "group", value_col]
    )

    if temp.empty:
        print(f"No valid observations for {value_col}.")
        return

    # Use the displayed group labels directly as hue values.
    # This keeps legend colors exactly aligned with the boxplot colors.
    temp["group_label"] = temp["group"].map(GROUP_LABEL_MAP)

    plt.figure(figsize=(12, 6))

    ax = sns.boxplot(
        data=temp,
        x="tier",
        y=value_col,
        hue="group_label",
        hue_order=[
            "ID1: Top 10%",
            "ID2: Remaining 90%",
        ],
        showfliers=False,
    )

    ax.set_xlabel("Tier")
    ax.set_ylabel(ylabel)

    if title is None:
        title = f"{ylabel} by Tier"

    ax.set_title(title)
    ax.legend(title="Group")

    plt.tight_layout()

    if output_name is None:
        output_name = f"boxplot_{value_col}.png"

    plt.savefig(
        os.path.join(OUTPUT_DIR, output_name),
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close()


## 8. Tier-level boxplots for original and additional indicators


In [ ]:
# Variables for tier-level ID1 vs ID2 boxplots
#
# The variables below include:
# 1. the original indicators;
# 2. the additional important variables supplied in the latest list;
# 3. kum_score.
#
# Note: occup_cd is categorical and is therefore kept in the categorical
# distribution section rather than being plotted as a boxplot.

boxplot_variables = {
    # Original indicators
    "pre_credit_limit": "Pre-approved Credit Limit",
    "lum_score": "LUM Score",
    "aum_score": "AUM Score",
    "yr_acm_mpb_fncltx_amt": "Annual Mobile Banking Transaction Amount",
    "yr_acm_mpb_land_monum": "Annual Mobile Banking Login Months",
    "acgmoclrr6mampbftxamt": "Transaction Amount over the Past 6 Months",
    "days_since_become_cust": "成为我行客户时长（天）",

    # Additional important variables
    "moacm_mpb_fncltx_dnum": "MOACM MPB Financial Transaction Days",
    "mo_acm_mpb_land_days": "Monthly Mobile Banking Login Days",
    "prvtcstrt12meccsmdnum": "PRVTCSTRT12MECCSMDNUM",
    "yracm_mpb_fncltx_dnum": "Annual Mobile Banking Financial Transaction Days",
    "acgmocrr6mampbftxdnum": "Transaction Days over the Past 6 Months",
    "prvtcstrt12mlpfccmdnum": "PRVTCSTRT12MLPFCCMDNUM",
    "prvtcstrt12mfsccnmpamt": "PRVTCSTRT12MFSCCNMPAMT",
    "prvtcstrt12mgsccsmdnum": "PRVTCSTRT12MGSCCSMDNUM",
    "kum_score": "KUM Score",
}

for variable, ylabel in boxplot_variables.items():
    plot_boxplot_by_tier(
        df=plot_df,
        value_col=variable,
        ylabel=ylabel,
        title=f"{ylabel} by Tier: ID1 vs ID2",
        output_name=f"boxplot_{variable}.png",
    )


## Current LUM by Tier: F1, F2, and F3 only

This figure compares ID1 and ID2 for `cur_lum` using only tiers F1, F2, and F3. Tiers D and E are excluded from this plot.


In [ ]:
# ============================================================
# Current LUM boxplot: F1 / F2 / F3 only
# ============================================================

cur_lum_col = "cur_lum"
selected_tiers = ["F1", "F2", "F3"]

if cur_lum_col not in plot_df.columns:
    print(f"Skip {cur_lum_col}; column not found.")

else:
    cur_lum_df = plot_df[
        plot_df["tier"].astype(str).str.upper().isin(selected_tiers)
    ][["tier", "group", cur_lum_col]].copy()

    cur_lum_df["tier"] = (
        cur_lum_df["tier"]
        .astype(str)
        .str.upper()
    )

    cur_lum_df[cur_lum_col] = pd.to_numeric(
        cur_lum_df[cur_lum_col],
        errors="coerce"
    )

    cur_lum_df = cur_lum_df.dropna(
        subset=["tier", "group", cur_lum_col]
    )

    cur_lum_df["group_label"] = cur_lum_df["group"].map(
        GROUP_LABEL_MAP
    )

    plt.figure(figsize=(10, 6))

    ax = sns.boxplot(
        data=cur_lum_df,
        x="tier",
        y=cur_lum_col,
        hue="group_label",
        order=["F1", "F2", "F3"],
        hue_order=[
            "ID1: Top 10%",
            "ID2: Remaining 90%",
        ],
        showfliers=False,
    )

    ax.set_xlabel("Tier")
    ax.set_ylabel("Current LUM")
    ax.set_title("Current LUM by Tier: ID1 vs ID2")
    ax.legend(title="Group")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            "boxplot_cur_lum_F1_F2_F3.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


## Binary indicator distribution: ID1 vs ID2

The variable `lblclbrmpbactvcst_ind` takes only values 0 and 1, so its distribution is shown using separate pie charts for ID1 and ID2.


In [ ]:
# ============================================================
# Pie charts for lblclbrmpbactvcst_ind: ID1 vs ID2
# ============================================================

indicator_col = "lblclbrmpbactvcst_ind"

if indicator_col not in plot_df.columns:
    print(f"Skip {indicator_col}; column not found.")

else:
    pie_df = plot_df[
        plot_df["group"].isin(["id1", "id2"])
    ][["group", indicator_col]].copy()

    # Convert to numeric and keep only valid binary values 0 and 1
    pie_df[indicator_col] = pd.to_numeric(
        pie_df[indicator_col],
        errors="coerce"
    )

    pie_df = pie_df[
        pie_df[indicator_col].isin([0, 1])
    ].copy()

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10, 5)
    )

    group_info = [
        ("id1", "ID1: Top 10%"),
        ("id2", "ID2: Remaining 90%"),
    ]

    for ax, (group_key, group_title) in zip(axes, group_info):

        group_values = (
            pie_df.loc[
                pie_df["group"] == group_key,
                indicator_col
            ]
            .value_counts()
            .reindex([0, 1], fill_value=0)
        )

        total = int(group_values.sum())

        if total == 0:
            ax.text(
                0.5,
                0.5,
                "No valid observations",
                ha="center",
                va="center"
            )
            ax.set_title(group_title)
            ax.axis("off")
            continue

        ax.pie(
            group_values.values,
            labels=["Value = 0", "Value = 1"],
            autopct="%1.1f%%",
            startangle=90,
            counterclock=False
        )

        ax.set_title(
            f"{group_title}\n"
            f"{indicator_col} Distribution"
        )

        ax.axis("equal")

    fig.suptitle(
        f"{indicator_col}: ID1 vs ID2",
        fontsize=14
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"pie_{indicator_col}_id1_id2.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    # Also display the counts and within-group percentages
    indicator_summary = (
        pie_df
        .groupby("group")[indicator_col]
        .value_counts()
        .rename("count")
        .reset_index()
    )

    indicator_summary["percentage"] = (
        indicator_summary["count"]
        / indicator_summary.groupby("group")["count"].transform("sum")
        * 100
    )

    display(indicator_summary)


## 9. Extract ID1 and ID2 samples

In [ ]:
id1_df = plot_df[
    plot_df["group"] == "id1"
].copy()

id2_df = plot_df[
    plot_df["group"] == "id2"
].copy()

print("id1_df shape:", id1_df.shape)
print("id2_df shape:", id2_df.shape)


## 10. Function: categorical distribution for a single group

In [ ]:
DISPLAY_NAME_MAP = {
    "tier": "Tier",
    "age": "Age",
    "education_cd": "Education",
    "occup_cd": "Occupation",
    "gnd_cd": "Gender",
    "pre_credit_limit": "Pre-approved Credit Limit",
}

def plot_categorical_distribution(
    df,
    col,
    title=None,
    output_name=None,
):
    if col not in df.columns:
        print(f"Skip {col}; column not found.")
        return

    temp = df[col].fillna("Missing").astype(str)

    freq = (
        temp.value_counts(normalize=True)
        .mul(100)
        .rename("percentage")
        .reset_index()
        .rename(columns={"index": col})
    )

    # Compatibility with newer pandas versions
    if col not in freq.columns:
        freq.columns = [col, "percentage"]

    plt.figure(figsize=(10, 6))

    ax = sns.barplot(
        data=freq,
        x=col,
        y="percentage",
    )

    xlabel = DISPLAY_NAME_MAP.get(col, col)

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Share (%)")

    if title is None:
        title = f"{xlabel} Distribution"

    ax.set_title(title)
    plt.xticks(rotation=45, ha="right")

    for container in ax.containers:
        ax.bar_label(
            container,
            fmt="%.1f%%",
            padding=3,
        )

    plt.tight_layout()

    if output_name is None:
        output_name = f"{col}_distribution.png"

    plt.savefig(
        os.path.join(OUTPUT_DIR, output_name),
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close()


## Education score distribution

`education_cd` is treated as an ordered numeric score rather than a string category. The x-axis is therefore sorted numerically from low to high.


In [ ]:
# ============================================================
# Education score distribution
# ============================================================

education_col = "education_cd"

if education_col not in plot_df.columns:
    print(f"Skip {education_col}; column not found.")

else:
    # --------------------------------------------------------
    # Prepare numeric education scores
    # --------------------------------------------------------
    education_df = plot_df[
        [ID_COL, "group", education_col]
    ].copy()

    education_df[education_col] = pd.to_numeric(
        education_df[education_col],
        errors="coerce"
    )

    education_df = education_df.dropna(
        subset=[education_col]
    )

    # Use one row per customer
    education_df = education_df.drop_duplicates(
        subset=[ID_COL],
        keep="first"
    )

    education_order = sorted(
        education_df[education_col].unique()
    )

    # --------------------------------------------------------
    # A. Education distribution within ID1
    # --------------------------------------------------------
    id1_education = education_df[
        education_df["group"] == "id1"
    ].copy()

    id1_education_freq = (
        id1_education[education_col]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .rename("percentage")
        .reset_index()
    )

    id1_education_freq.columns = [
        education_col,
        "percentage"
    ]

    fig, ax = plt.subplots(figsize=(10, 6))

    bars = ax.bar(
        id1_education_freq[education_col].astype(str),
        id1_education_freq["percentage"]
    )

    for bar, pct in zip(
        bars,
        id1_education_freq["percentage"]
    ):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{pct:.1f}%",
            ha="center",
            va="bottom"
        )

    ax.set_xlabel("Education Score")
    ax.set_ylabel("Share (%)")
    ax.set_title("Education Score Distribution of ID1 Customers")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            "id1_education_score_distribution.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    # --------------------------------------------------------
    # B. Education distribution: ID1 vs ID2
    # --------------------------------------------------------
    education_group_freq = (
        education_df
        .groupby("group")[education_col]
        .value_counts(normalize=True)
        .mul(100)
        .rename("percentage")
        .reset_index()
    )

    # Pivot to ensure a consistent numeric x-axis order
    education_pivot = (
        education_group_freq
        .pivot(
            index=education_col,
            columns="group",
            values="percentage"
        )
        .reindex(education_order)
        .fillna(0)
    )

    for col in ["id1", "id2"]:
        if col not in education_pivot.columns:
            education_pivot[col] = 0

    education_pivot = education_pivot[["id1", "id2"]]

    x = np.arange(len(education_pivot.index))
    width = 0.38

    fig, ax = plt.subplots(figsize=(11, 6))

    bars1 = ax.bar(
        x - width / 2,
        education_pivot["id1"].values,
        width,
        label="ID1: Top 10%"
    )

    bars2 = ax.bar(
        x + width / 2,
        education_pivot["id2"].values,
        width,
        label="ID2: Remaining 90%"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [str(v) for v in education_pivot.index]
    )

    ax.set_xlabel("Education Score")
    ax.set_ylabel("Within-group Share (%)")
    ax.set_title("Education Score Distribution: ID1 vs ID2")
    ax.legend(title="Group")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            "id1_id2_education_score_distribution.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    # --------------------------------------------------------
    # Display the underlying ordered statistics
    # --------------------------------------------------------
    display(
        education_pivot
        .rename(
            columns={
                "id1": "ID1: Top 10%",
                "id2": "ID2: Remaining 90%"
            }
        )
    )


## 11. ID1 categorical distribution: Occupation

Tier, age, and gender are analyzed separately using ID1 selection rates, which adjust for differences in the original group sizes.


In [ ]:
id1_categorical_variables = [
    "occup_cd",
]

for col in id1_categorical_variables:
    label = DISPLAY_NAME_MAP.get(col, col)

    plot_categorical_distribution(
        df=id1_df,
        col=col,
        title=f"{label} Distribution of ID1 Customers",
        output_name=f"id1_{col}_distribution.png",
    )


## 12. Function: ID1 vs ID2 categorical distribution

In [ ]:
def plot_group_categorical_distribution(
    df,
    col,
    title=None,
    output_name=None,
):
    if col not in df.columns:
        print(f"Skip {col}; column not found.")
        return

    temp = df[["group", col]].copy()
    temp[col] = temp[col].fillna("Missing").astype(str)
    temp["group_label"] = temp["group"].map(GROUP_LABEL_MAP)

    freq = (
        temp.groupby("group_label")[col]
        .value_counts(normalize=True)
        .mul(100)
        .rename("percentage")
        .reset_index()
    )

    plt.figure(figsize=(11, 6))

    ax = sns.barplot(
        data=freq,
        x=col,
        y="percentage",
        hue="group_label",
        hue_order=[
            "ID1: Top 10%",
            "ID2: Remaining 90%",
        ],
    )

    xlabel = DISPLAY_NAME_MAP.get(col, col)

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Within-group Share (%)")

    if title is None:
        title = f"{xlabel} Distribution: ID1 vs ID2"

    ax.set_title(title)
    plt.xticks(rotation=45, ha="right")
    ax.legend(title="Group")

    plt.tight_layout()

    if output_name is None:
        output_name = f"id1_id2_{col}_distribution.png"

    plt.savefig(
        os.path.join(OUTPUT_DIR, output_name),
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close()


## 13. Function: ID1 vs ID2 continuous distribution

In [ ]:
def plot_group_continuous_distribution(
    df,
    col,
    xlabel,
    title=None,
    output_name=None,
    bins=30,
):
    if col not in df.columns:
        print(f"Skip {col}; column not found.")
        return

    temp = df[["group", col]].copy()

    temp[col] = pd.to_numeric(
        temp[col],
        errors="coerce",
    )

    temp = temp.dropna(subset=[col])

    if temp.empty:
        print(f"No valid observations for {col}.")
        return

    fig, ax = plt.subplots(figsize=(10, 6))

    # Use matplotlib hist instead of sns.histplot for compatibility
    # with older seaborn versions.
    for group_key, group_label in [
        ("id1", "ID1: Top 10%"),
        ("id2", "ID2: Remaining 90%"),
    ]:
        values = temp.loc[
            temp["group"] == group_key,
            col
        ].dropna()

        if len(values) == 0:
            continue

        ax.hist(
            values,
            bins=bins,
            density=True,
            histtype="step",
            linewidth=2,
            label=group_label
        )

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")

    if title is None:
        title = f"{xlabel} Distribution: ID1 vs ID2"

    ax.set_title(title)
    ax.legend(title="Group")

    plt.tight_layout()

    if output_name is None:
        output_name = f"id1_id2_{col}_distribution.png"

    plt.savefig(
        os.path.join(OUTPUT_DIR, output_name),
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close()


## 14. Overall distribution comparison for remaining variables

Occupation is compared using within-group shares. Education score is handled separately using an ordered numeric axis, and pre-approved credit limit is compared using density distributions. Tier, age, and gender are handled by the selection-rate plots above.


In [ ]:
group_categorical_variables = [
    "occup_cd",
]

for col in group_categorical_variables:
    label = DISPLAY_NAME_MAP.get(col, col)

    plot_group_categorical_distribution(
        df=plot_df,
        col=col,
        title=f"{label} Distribution: ID1 vs ID2",
    )

plot_group_continuous_distribution(
    df=plot_df,
    col="pre_credit_limit",
    xlabel="Pre-approved Credit Limit",
    title="Pre-approved Credit Limit Distribution: ID1 vs ID2",
    output_name="id1_id2_pre_credit_limit_distribution.png",
    bins=30,
)


## 15. Descriptive statistics

In [ ]:
print("=" * 60)
print("ID1 / ID2 Sample Summary")
print("=" * 60)

print("\nID1:")
print(id1_df.shape)

print("\nID2:")
print(id2_df.shape)

summary_cols = [
    "age",
    "pre_credit_limit",
    "lum_score",
    "aum_score",
    "yr_acm_mpb_fncltx_amt",
    "yr_acm_mpb_land_monum",
    "acgmoclrr6mampbftxamt",
]

summary_cols = [
    col
    for col in summary_cols
    if col in plot_df.columns
]

if summary_cols:
    summary = (
        plot_df
        .groupby("group")[summary_cols]
        .describe()
    )

    summary_path = os.path.join(
        OUTPUT_DIR,
        "id1_id2_summary.xlsx",
    )

    summary.to_excel(summary_path)

    print(f"\nDescriptive statistics saved to: {summary_path}")
    display(summary)

print(f"\nAll figures are saved to: {OUTPUT_DIR}")
